# Create CF-compliant NetCDF from Oslofjord marine water chemistry dataset

Converts `oslofjord_cleaned_standardized_data_2025v1.csv` (multiple stations, discrete depth profiles) to a NetCDF-4 file following CF-1.13 / ACDD-1.3 conventions.

**CF feature type:** `timeSeriesProfile`  
**CF representation:** H.5.3 – Ragged array representation of time series profiles  

Dimensions:
- `station` – one entry per fixed monitoring station
- `profile` – one entry per (station × date) pair; indexed into `station` via `stationIndex`
- `obs`     – one entry per individual measurement; grouped into profiles via `rowSize`

Counting variables:
- `stationIndex(profile)` → `instance_dimension = "station"` (indexed ragged array)
- `rowSize(profile)` → `sample_dimension = "obs"` (contiguous ragged array)

## 1. Imports

In [12]:
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import netCDF4 as nc


## 2. Load metadata

In [13]:
meta_path = Path("../hydrochem_trends_river_oslofjord_use_case/config/marine/oslofjord.json")
with meta_path.open() as f:
    meta = json.load(f)

print(f"Loaded metadata for {len(meta['variables'])} variables")


Loaded metadata for 20 variables


## 3. Load CSV

In [14]:
cfg = meta["input"]
df = pd.read_csv(Path(f"../{cfg['file']}"), parse_dates=[cfg["time_col"]])

time_col  = cfg["time_col"]
depth_col = cfg["depth_col"]
stn_col   = cfg["station_col"]
code_col  = cfg["station_code_col"]
lat_col   = cfg["latitude_col"]
lon_col   = cfg["longitude_col"]

# Sort so obs within each profile are depth-ordered
df = df.sort_values([stn_col, time_col, depth_col]).reset_index(drop=True)

print(f"Rows: {len(df)},  Stations: {df[stn_col].nunique()},  "
      f"Date range: {df[time_col].min().date()} – {df[time_col].max().date()}")


Rows: 74895,  Stations: 70,  Date range: 1990-01-08 – 2025-12-07


## 4. Build ragged-array index structure

Group rows into *profiles* (unique station × date), then into *stations*.

In [15]:
stations = (
    df[[stn_col, code_col, lat_col, lon_col]]
    .drop_duplicates(subset=[stn_col])
    .reset_index(drop=True)
)
station_index_map = {name: i for i, name in enumerate(stations[stn_col])}

df["_profile_key"] = list(zip(df[stn_col], df[time_col].dt.date))

profile_info = (
    df.groupby("_profile_key", sort=False)
    .agg(
        station_name=(stn_col, "first"),
        time=(time_col, "first"),
        n_obs=(depth_col, "count"),
    )
    .reset_index(drop=True)
)

# stationIndex maps each profile → its station index
stationIndex = profile_info["station_name"].map(station_index_map).values.astype(np.int32)
# rowSize is the number of depth observations in each profile
rowSize      = profile_info["n_obs"].values.astype(np.int32)
profile_times = profile_info["time"].values  # numpy datetime64

print(f"Stations: {len(stations)}  Profiles: {len(profile_info)}  Obs: {rowSize.sum()}")
assert rowSize.sum() == len(df), "obs count mismatch"


Stations: 70  Profiles: 13088  Obs: 74895


## 5. Build observation-level arrays

In [16]:
def _sanitize_varname(name: str) -> str:
    """Replace characters problematic for THREDDS/NetCDF variable names."""
    return (
        name
        .replace("µ", "u")
        .replace(":", "_to_")
        .replace("+", "_plus_")
        .replace("-", "_")
    )

df = df.rename(columns={col: _sanitize_varname(col) for col in df.columns})

coord_cols = {stn_col, code_col, lat_col, lon_col, time_col, depth_col, "_profile_key"}
data_cols  = [c for c in df.columns if c not in coord_cols]

obs_depth = df[depth_col].values.astype(np.float32)
obs_data  = {col: df[col].values.astype(np.float32) for col in data_cols}

print("Data variables:", data_cols)


Data variables: ['CDOM_abs_443_1perm', 'Chl_a_ugperl', 'DIN_ugperl', 'DIN_to_DIP', 'DOC_mgperl', 'DOC_to_DIN', 'DOC_to_TOTN', 'NH4_N_ugperl', 'NO2_N_ugperl', 'NO3_plus_NO2_N_ugperl', 'NO3_N_ugperl', 'PO4_P_ugperl', 'SSS_PSU', 'SST_degC', 'TOC_mgperl', 'TOTN_ugperl', 'TOTN_to_TOTP', 'TOTP_ugperl', 'TSM_mgperl', 'secchi_depth_m']


## 6. Write CF-compliant NetCDF (H.5.3 ragged array)

Station names may contain non-ASCII characters (Norwegian), so we use NetCDF-4 variable-length UTF-8 strings (`str` type) for `station_name` and `station_code`.

In [17]:
out_cfg  = meta["output"]
out_path = Path("..") / out_cfg["filename"]
out_path.parent.mkdir(parents=True, exist_ok=True)
gm = meta["global_metadata"]
vm = meta["variables"]

# Convert profile times to integer seconds since 1970-01-01
epoch = np.datetime64("1970-01-01", "s")
profile_time_int = (profile_times.astype("datetime64[s]") - epoch).astype(np.int32)

with nc.Dataset(out_path, "w", format=out_cfg["format"]) as ds:

    # ---- dimensions ------------------------------------------------
    ds.createDimension("station", len(stations))
    ds.createDimension("profile", len(profile_info))
    ds.createDimension("obs",     int(rowSize.sum()))

    # ---- station-level variables ------------------------------------
    lat_var = ds.createVariable("lat", "f4", ("station",), fill_value=False)
    lat_var.standard_name = "latitude"
    lat_var.long_name     = "Station Latitude"
    lat_var.units         = "degrees_north"
    lat_var[:]            = stations[lat_col].values.astype(np.float32)

    lon_var = ds.createVariable("lon", "f4", ("station",), fill_value=False)
    lon_var.standard_name = "longitude"
    lon_var.long_name     = "Station Longitude"
    lon_var.units         = "degrees_east"
    lon_var[:]            = stations[lon_col].values.astype(np.float32)

    # vlen str supports non-ASCII (Norwegian) characters
    stn_var = ds.createVariable("station_name", str, ("station",))
    stn_var.long_name = "Station Name"
    stn_var.cf_role   = "timeseries_id"
    stn_var[:]        = stations[stn_col].values

    code_var = ds.createVariable("station_code", str, ("station",))
    code_var.long_name = "Station Code"
    code_var[:]        = stations[code_col].values

    # ---- profile-level variables (H.5.3 counting variables) --------
    # stationIndex: maps each profile to its station (indexed ragged array)
    si_var = ds.createVariable("stationIndex", "i4", ("profile",), fill_value=False)
    si_var.long_name          = "Index of station this profile belongs to"
    si_var.instance_dimension = "station"
    si_var[:]                 = stationIndex

    # rowSize: number of obs per profile (contiguous ragged array)
    rs_var = ds.createVariable("rowSize", "i4", ("profile",), fill_value=False)
    rs_var.long_name        = "Number of observations in this profile"
    rs_var.sample_dimension = "obs"
    rs_var[:]               = rowSize

    # time: one timestamp per profile
    t_cfg = out_cfg["time"]
    t_var = ds.createVariable("time", "i4", ("profile",), fill_value=False)
    t_var.standard_name = "time"
    t_var.long_name     = "Time of profile"
    t_var.units         = t_cfg["units"]
    t_var.calendar      = t_cfg["calendar"]
    t_var[:]            = profile_time_int

    # ---- obs-level variables ----------------------------------------
    z_var = ds.createVariable("depth", "f4", ("obs",))
    z_var.standard_name = "depth"
    z_var.long_name     = "Depth of observation"
    z_var.units         = "m"
    z_var.positive      = "down"
    z_var[:]            = obs_depth

    coord_str = "time lat lon depth station_name"
    for col in data_cols:
        v = ds.createVariable(col, "f4", ("obs",))
        v.coordinates = coord_str
        if col in vm:
            for attr in ("standard_name", "short_name", "long_name", "units",
                         "units_metadata", "comment"):
                if attr in vm[col]:
                    setattr(v, attr, vm[col][attr])
        v[:] = obs_data[col]

    # ---- global attributes ------------------------------------------
    for k, v in gm.items():
        setattr(ds, k, v)
    ds.time_coverage_start          = pd.Timestamp(profile_times.min()).strftime("%Y-%m-%dT%H:%M:%SZ")
    ds.time_coverage_end            = pd.Timestamp(profile_times.max()).strftime("%Y-%m-%dT%H:%M:%SZ")
    ds.geospatial_lat_min           = float(stations[lat_col].min())
    ds.geospatial_lat_max           = float(stations[lat_col].max())
    ds.geospatial_lon_min           = float(stations[lon_col].min())
    ds.geospatial_lon_max           = float(stations[lon_col].max())
    ds.geospatial_vertical_min      = float(obs_depth.min())
    ds.geospatial_vertical_max      = float(obs_depth.max())
    ds.geospatial_vertical_positive = "down"
    ds.date_created                 = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Written: {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")


Written: ../hydrochem_trends_river_oslofjord_use_case/data/raw/marine/oslofjord_cleaned_standardized_data.nc  (6.5 MB)


---
## 8. Selecting data with xarray

The H.5.3 ragged structure stores every observation in flat `obs`-indexed arrays. Open the file and expand the counting variables into per-obs lookup arrays once; then every selection is a simple boolean mask.

In [18]:
import xarray as xr
import numpy as np
import pandas as pd

ds = xr.open_dataset(Path(f"../{meta['output']['filename']}"))

# Expand ragged structure: for each obs → which station/time does it belong to?
profile_of_obs = np.repeat(np.arange(ds.dims["profile"]), ds["rowSize"].values)
stn_of_obs  = ds["stationIndex"].values[profile_of_obs]
time_of_obs = ds["time"].values[profile_of_obs]
depth_of_obs = ds["depth"].values
stns = ds["station_name"].values


/tmp/ipykernel_10346/3209659499.py:8: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  profile_of_obs = np.repeat(np.arange(ds.dims["profile"]), ds["rowSize"].values)


### Example 1 – All observations for a single station

In [21]:
mask = stn_of_obs == np.where(stns == "Arendal")[0][0]

pd.DataFrame({
    "time":    time_of_obs[mask],
    "depth_m": depth_of_obs[mask],
    "SST_degC":    ds["SST_degC"].values[mask],
    "Chl-a_µgperl": ds["Chl_a_ugperl"].values[mask],
})


,time,depth_m,SST_degC,Chl-a_µgperl
0,1990-01-08,0.0,6.01,0.89
1,1990-01-08,5.0,6.08,0.63
2,1990-01-08,10.0,6.33,0.51
3,1990-02-04,0.0,5.82,0.60
4,1990-02-04,5.0,5.83,0.51
...,...,...,...,...
2237,2024-11-08,5.0,11.37,0.49
2238,2024-11-08,10.0,12.01,0.34
2239,2024-12-10,0.0,NaN,NaN
2240,2024-12-10,5.0,NaN,NaN


### Example 2 – Single station, surface only (depth = 0 m)

In [22]:
mask = (stn_of_obs == np.where(stns == "Arendal")[0][0]) & (depth_of_obs == 0)

pd.DataFrame({
    "time":         time_of_obs[mask],
    "SST_degC":     ds["SST_degC"].values[mask],
    "SSS_PSU":      ds["SSS_PSU"].values[mask],
    "Chl-a_µgperl": ds["Chl_a_ugperl"].values[mask],
})


,time,SST_degC,SSS_PSU,Chl-a_µgperl
0,1990-01-08,6.010000,30.410000,0.89
1,1990-02-04,5.820000,32.500000,0.60
2,1990-03-01,5.380000,30.879999,0.76
3,1990-04-03,6.380000,32.990002,0.48
4,1990-04-17,6.480000,27.770000,2.52
...,...,...,...,...
686,2024-08-06,18.520000,29.350000,1.07
687,2024-09-14,16.690001,26.799999,4.12
688,2024-10-09,13.800000,23.490000,2.57
689,2024-11-08,10.240000,26.170000,1.17


### Example 3 – Single station, specific time window

In [23]:
mask = (
    (stn_of_obs == np.where(stns == "Bastø")[0][0])
    & (time_of_obs >= pd.Timestamp("2010-01-01").to_datetime64())
    & (time_of_obs <  pd.Timestamp("2020-01-01").to_datetime64())
)

pd.DataFrame({
    "time":       time_of_obs[mask],
    "depth_m":    depth_of_obs[mask],
    "TOTN_µgperl": ds["TOTN_ugperl"].values[mask],
    "Chl-a_µgperl": ds["Chl_a_ugperl"].values[mask],
})


,time,depth_m,TOTN_µgperl,Chl-a_µgperl
0,2010-03-20,0.0,251.25,5.96
1,2010-03-20,1.0,NaN,NaN
2,2010-03-20,2.0,NaN,NaN
3,2010-03-20,3.0,NaN,NaN
4,2010-03-20,4.0,NaN,NaN
...,...,...,...,...
687,2019-12-10,6.5,NaN,NaN
688,2019-12-10,7.5,NaN,NaN
689,2019-12-10,8.5,NaN,NaN
690,2019-12-10,9.5,NaN,NaN


### Example 4 – Multiple stations, surface, summer months (June–August)

In [24]:
selected = ["Arendal", "Bastø", "Steilene"]
mask = (
    np.isin(stn_of_obs, np.where(np.isin(stns, selected))[0])
    & (depth_of_obs == 0)
    & np.isin(pd.DatetimeIndex(time_of_obs).month, [6, 7, 8])
)

pd.DataFrame({
    "station":      stns[stn_of_obs[mask]],
    "time":         time_of_obs[mask],
    "Chl-a_µgperl": ds["Chl_a_ugperl"].values[mask],
})


,station,time,Chl-a_µgperl
0,Arendal,1990-06-14,0.70
1,Arendal,1990-06-25,0.77
2,Arendal,1990-07-18,1.26
3,Arendal,1990-08-01,1.63
4,Arendal,1990-08-30,0.84
...,...,...,...
484,Steilene,2024-06-24,7.30
485,Steilene,2024-07-09,0.35
486,Steilene,2024-07-23,1.80
487,Steilene,2024-08-05,1.00


### Example 5 – Depth slice across all stations (5 m), station-level summary

In [30]:
mask = np.isclose(depth_of_obs, 5.0)

pd.DataFrame({
    "station":    stns[stn_of_obs[mask]],
    "NO3_N_ugperl": ds["NO3_N_ugperl"].values[mask],
}).groupby("station")["NO3_N_ugperl"].describe().round(2).dropna()


,count,mean,std,min,25%,50%,75%,max
station,,,,,,,,
Arendal,94.0,24.53,34.19,0.10,1.55,4.27,43.14,153.00
Bastø,116.0,45.82,45.08,0.14,4.89,26.00,79.77,166.31
Breiangen vest,179.0,50.54,48.06,0.14,8.68,28.29,87.68,160.66
Filtvedt,54.0,62.61,58.31,0.56,9.64,49.59,103.30,238.67
Frierfjorden,53.0,113.87,49.48,31.16,69.02,116.54,151.83,235.76
Håøyfjorden,65.0,40.78,44.50,0.84,2.30,17.57,74.31,152.18
Indre Drammensfjord Solumstranda,33.0,244.66,81.24,118.36,177.66,230.27,302.82,399.27
Kippenes,38.0,67.14,49.94,0.10,21.73,61.49,109.58,145.81
Langesundsfjorden,79.0,45.49,44.91,0.56,4.87,31.13,78.83,155.60
